# Step 3: Replikasi Murni dari Paper (Tanpa Koreksi Metodologi)

Notebook ini SENGAJA mereplikasi kemungkinan pipeline preprocessing paper APA ADANYA, tanpa
koreksi anti-leakage apa pun -- kebalikan dari `04_preprocessing_split.ipynb` dan
`05_modeling_evaluation.ipynb` yang mengimplementasikan pipeline TERKOREKSI (Tugas #2).

**Tujuan:** mengkuantifikasi seberapa besar dampak dua bug leakage yang diidentifikasi di
`AUDIT_METODOLOGI.md`, dengan membandingkan hasil model pada versi replikasi murni (notebook ini) vs versi
terkoreksi (notebook 04-05) vs paper asli.

**Definisi "replikasi murni" di sini** (disepakati dengan user, 17 Sep 2026):
- Seleksi fitur chi-square (top-20) DAN standard scaling di-fit ke **SELURUH dataset** (12.330
  baris, termasuk yang nantinya jadi test set), SEBELUM split 70/30.
- Split 70/30 (stratified, random_state=42 -- identik dengan notebook 04) dilakukan SETELAH
  chi2+scaling, supaya train dan test sama-sama sudah "melihat" preprocessing yang di-fit dari
  seluruh data.
- SMOTE tetap HANYA di train, sesudah split -- ini konsisten dengan versi terkoreksi, karena
  paper menyatakan eksplisit SMOTE hanya di train (satu-satunya bagian pipeline yang paper
  jelaskan urutannya dengan pasti, jadi tidak ada ambiguitas untuk direplikasi "replikasi murni").
- Parameter model: identik dengan notebook 05 (parameter yang dilaporkan paper, tanpa tuning).

Catatan: ini BUKAN rekomendasi cara kerja yang benar -- notebook ini eksperimen kontrol yang
sengaja mereplikasi kemungkinan kesalahan metodologis paper untuk tujuan perbandingan, bukan
pipeline yang dipakai untuk kesimpulan akhir Tugas #2/#3.


In [1]:
import sys, os
sys.path.append('../src')
from data_utils import load_data
from models import get_models
from evaluation import evaluate_model
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.metrics import precision_score, f1_score

os.makedirs('../results/tables', exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.3
CHI2_K = 20

NUMERIC_FEATURES = [
    'Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration',
    'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay',
]
RAW_CODE_FEATURES = ['OperatingSystems', 'Browser', 'Region', 'TrafficType']
BINARY_FEATURE = 'Weekend'
ONEHOT_FEATURES = ['Month', 'VisitorType']
TARGET = 'Revenue'

df = load_data()
print('Dataset asli:', df.shape)


Dataset asli: (12330, 18)


## 1. Encoding, Seleksi Fitur Chi-Square, dan Scaling -- di-fit ke SELURUH Dataset

Berbeda dengan notebook 04 (encoder/selector/scaler di-fit HANYA di train, SETELAH split), di sini
ketiganya di-fit ke `df` penuh (12.330 baris) SEBELUM split dilakukan sama sekali. Ini persis
skenario leakage yang diperingatkan `AUDIT_METODOLOGI.md`: keputusan preprocessing (fitur mana yang
signifikan secara chi2, parameter scaling apa) ikut ditentukan oleh baris yang nantinya jadi test
set.


In [2]:
X_full = df.drop(columns=[TARGET])
y_full = df[TARGET].astype(int)

# Encoding (fit di SELURUH data)
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(X_full[ONEHOT_FEATURES])
ohe_full = pd.DataFrame(
    encoder.transform(X_full[ONEHOT_FEATURES]),
    columns=encoder.get_feature_names_out(ONEHOT_FEATURES), index=X_full.index,
)
passthrough_cols = NUMERIC_FEATURES + RAW_CODE_FEATURES + [BINARY_FEATURE]
pass_full = X_full[passthrough_cols].copy()
pass_full[BINARY_FEATURE] = pass_full[BINARY_FEATURE].astype(int)
X_enc_full = pd.concat([pass_full.reset_index(drop=True), ohe_full.reset_index(drop=True)], axis=1)

print('Jumlah fitur setelah encoding (seluruh dataset):', X_enc_full.shape[1])

# Seleksi fitur chi2 (fit di SELURUH data, termasuk label y_full -- LEAKAGE: keputusan fitur
# ikut ditentukan baris yang nanti jadi test set)
assert (X_enc_full.values >= 0).all(), 'Semua fitur harus non-negatif untuk chi2'
selector = SelectKBest(score_func=chi2, k=CHI2_K)
selector.fit(X_enc_full, y_full.reset_index(drop=True))
selected_cols = X_enc_full.columns[selector.get_support()].tolist()
print(f'Jumlah fitur terpilih (dari SELURUH dataset): {len(selected_cols)}')
print('Fitur terpilih:', selected_cols)

X_sel_full = X_enc_full[selected_cols].copy()

# Scaling (fit di SELURUH data -- LEAKAGE: parameter mean/std ikut dipengaruhi baris test set)
numeric_present = [c for c in NUMERIC_FEATURES if c in selected_cols]
scaler = StandardScaler()
scaler.fit(X_sel_full[numeric_present])
X_sel_full[numeric_present] = scaler.transform(X_sel_full[numeric_present])

print()
print('Preprocessing (encoding + chi2 + scaling) selesai di-fit ke SELURUH dataset (replikasi murni).')
print('Shape data siap split:', X_sel_full.shape)


Jumlah fitur setelah encoding (seluruh dataset): 28
Jumlah fitur terpilih (dari SELURUH dataset): 20
Fitur terpilih: ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Browser', 'Weekend', 'Month_Dec', 'Month_Feb', 'Month_Mar', 'Month_May', 'Month_Nov', 'Month_Oct', 'VisitorType_New_Visitor', 'VisitorType_Returning_Visitor']

Preprocessing (encoding + chi2 + scaling) selesai di-fit ke SELURUH dataset (replikasi murni).
Shape data siap split: (12330, 20)


## 2. Split 70/30 (Setelah Preprocessing) -- SMOTE Hanya di Train

Split dilakukan SETELAH preprocessing di atas -- kebalikan urutan dari notebook 04. `random_state`
dan `test_size` sama seperti notebook 04 supaya proporsi train/test sebanding.


In [3]:
X_train_pure, X_test_pure, y_train_pure, y_test_pure = train_test_split(
    X_sel_full, y_full, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_full,
)

print(f'Train: {X_train_pure.shape}, Test: {X_test_pure.shape}')
print('Proporsi kelas positif -- train:', round(y_train_pure.mean(), 4),
      '| test:', round(y_test_pure.mean(), 4))

smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train_pure, y_train_pure)

print()
print('Sebelum SMOTE -- train:', X_train_pure.shape, '| distribusi:', y_train_pure.value_counts().to_dict())
print('Setelah SMOTE  -- train:', X_train_res.shape, '| distribusi:', y_train_res.value_counts().to_dict())


Train: (8631, 20), Test: (3699, 20)
Proporsi kelas positif -- train: 0.1548 | test: 0.1546

Sebelum SMOTE -- train: (8631, 20) | distribusi: {0: 7295, 1: 1336}
Setelah SMOTE  -- train: (14590, 20) | distribusi: {0: 7295, 1: 7295}


## 3. Training 5 Model (Parameter Sesuai Paper, Identik Notebook 05)


In [4]:
models = get_models()
results_pure = {}
trained_models_pure = {}

for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    y_pred = model.predict(X_test_pure)
    y_proba = model.predict_proba(X_test_pure)[:, 1]
    metrics = evaluate_model(y_test_pure, y_pred, y_proba)
    metrics['Precision (weighted)'] = precision_score(y_test_pure, y_pred, average='weighted', zero_division=0)
    metrics['F1-Score (weighted)'] = f1_score(y_test_pure, y_pred, average='weighted', zero_division=0)
    results_pure[name] = metrics
    trained_models_pure[name] = model
    print(f'{name}: Accuracy={metrics["Accuracy"]:.4f}, Precision(w)={metrics["Precision (weighted)"]:.4f}, F1(w)={metrics["F1-Score (weighted)"]:.4f}')

results_pure_df = pd.DataFrame(results_pure).T
results_pure_df = results_pure_df[['Accuracy', 'Precision', 'TPR (Recall)', 'F1-Score', 'TNR', 'MCC', 'auROC', 'auPR', 'Precision (weighted)', 'F1-Score (weighted)']].round(4)
results_pure_df.to_csv('../results/tables/pure_replication_model_results.csv')
results_pure_df


Decision Tree: Accuracy=0.8662, Precision(w)=0.8936, F1(w)=0.8753


/Users/vickymahfudy/Study/Term 2/Data Science/Replikasi_Paper/.venv/lib/python3.11/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


SVM: Accuracy=0.8640, Precision(w)=0.8829, F1(w)=0.8711


MLP: Accuracy=0.8551, Precision(w)=0.8887, F1(w)=0.8661


Random Forest: Accuracy=0.8829, Precision(w)=0.8911, F1(w)=0.8863
XGBoost: Accuracy=0.8835, Precision(w)=0.8936, F1(w)=0.8875


,Accuracy,Precision,TPR (Recall),F1-Score,TNR,MCC,auROC,auPR,Precision (weighted),F1-Score (weighted)
Decision Tree,0.8662,0.5470,0.7832,0.6441,0.8814,0.5787,0.9132,0.6418,0.8936,0.8753
SVM,0.8640,0.5462,0.7133,0.6187,0.8916,0.5448,0.8693,0.6101,0.8829,0.8711
MLP,0.8551,0.5210,0.7815,0.6252,0.8686,0.5568,0.9044,0.6410,0.8887,0.8661
Random Forest,0.8829,0.6051,0.6993,0.6488,0.9165,0.5812,0.9135,0.6752,0.8911,0.8863
XGBoost,0.8835,0.6035,0.7185,0.6560,0.9137,0.5898,0.9225,0.7037,0.8936,0.8875


## 4. Perbandingan Tiga Arah: Paper vs Terkoreksi (Tugas #2) vs Replikasi Murni (Notebook Ini)

Fokus pada XGBoost dan Decision Tree (dua model yang dilaporkan paper secara detail).


In [5]:
corrected_df = pd.read_csv('../results/tables/model_results.csv', index_col=0)

paper_results = {
    'XGBoost': {
        'Accuracy': 0.9065, 'Precision': 0.9001, 'F1-Score': 0.898, 'TPR (Recall)': 0.801,
        'TNR': 0.96, 'auROC': 0.937, 'auPR': 0.749, 'MCC': 0.599,
    },
    'Decision Tree': {
        'Accuracy': 0.9054, 'Precision': 0.899, 'F1-Score': 0.8998, 'TPR (Recall)': 0.76,
        'TNR': 0.93, 'auROC': 0.923, 'auPR': 0.731, 'MCC': 0.605,
    },
}

metrics_common = ['Accuracy', 'Precision', 'TPR (Recall)', 'F1-Score', 'TNR', 'MCC', 'auROC', 'auPR']
rows = []
for model_name in ['XGBoost', 'Decision Tree']:
    for metric in metrics_common:
        rows.append({
            'Model': model_name,
            'Metric': metric,
            'Paper': paper_results[model_name][metric],
            'Terkoreksi (Tugas #2)': corrected_df.loc[model_name, metric],
            'Replikasi Murni (notebook ini)': results_pure_df.loc[model_name, metric],
        })

three_way = pd.DataFrame(rows)
three_way['Selisih Replikasi Murni - Terkoreksi'] = (three_way['Replikasi Murni (notebook ini)'] - three_way['Terkoreksi (Tugas #2)']).round(4)
three_way['Selisih Replikasi Murni - Paper'] = (three_way['Replikasi Murni (notebook ini)'] - three_way['Paper']).round(4)
three_way['Selisih Terkoreksi - Paper'] = (three_way['Terkoreksi (Tugas #2)'] - three_way['Paper']).round(4)
three_way.to_csv('../results/tables/pure_vs_corrected_vs_paper.csv', index=False)
three_way


,Model,Metric,Paper,Terkoreksi (Tugas #2),Replikasi Murni (notebook ini),Selisih Replikasi Murni - Terkoreksi,Selisih Replikasi Murni - Paper,Selisih Terkoreksi - Paper
0,XGBoost,Accuracy,0.9065,0.8867,0.8835,-0.0032,-0.0230,-0.0198
1,XGBoost,Precision,0.9001,0.6114,0.6035,-0.0079,-0.2966,-0.2887
2,XGBoost,TPR (Recall),0.8010,0.7343,0.7185,-0.0158,-0.0825,-0.0667
3,XGBoost,F1-Score,0.8980,0.6672,0.6560,-0.0112,-0.2420,-0.2308
4,XGBoost,TNR,0.9600,0.9146,0.9137,-0.0009,-0.0463,-0.0454
5,XGBoost,MCC,0.5990,0.6033,0.5898,-0.0135,-0.0092,0.0043
6,XGBoost,auROC,0.9370,0.9230,0.9225,-0.0005,-0.0145,-0.0140
7,XGBoost,auPR,0.7490,0.7051,0.7037,-0.0014,-0.0453,-0.0439
8,Decision Tree,Accuracy,0.9054,0.8662,0.8662,0.0000,-0.0392,-0.0392
9,Decision Tree,Precision,0.8990,0.5470,0.5470,0.0000,-0.3520,-0.3520


## 5. Diskusi: Seberapa Besar Dampak Bug Leakage Ini?

**Temuan utama:** untuk dataset dan pipeline ini, dampak leakage (chi2 + scaling di-fit ke seluruh
dataset sebelum split) TERNYATA KECIL -- jauh lebih kecil dari yang diperkirakan `AUDIT_METODOLOGI.md`
secara konseptual. Contoh XGBoost: Accuracy replikasi murni 0,8835 vs terkoreksi 0,8867 (selisih -0,0032),
MCC replikasi murni 0,5898 vs terkoreksi 0,6033 (selisih -0,0135) -- kecil, dan justru versi TERKOREKSI yang
sedikit lebih tinggi di kebanyakan metrik untuk XGBoost, bukan versi replikasi murni.

**Kenapa dampaknya kecil?** Dataset ini besar (12.330 baris, test set 30% = 3.699 baris). Statistik
yang di-fit dari seluruh dataset (12.330 baris) vs hanya train (8.631 baris) tidak banyak berbeda
secara numerik -- perbedaan antara "melihat 8.631 baris" vs "melihat 12.330 baris" untuk menghitung
mean/std atau skor chi2 relatif kecil ketika n sudah besar dan test set tidak systematically berbeda
distribusinya dari train (keduanya hasil random stratified split dari populasi yang sama).

**Implikasi:** temuan ini TIDAK berarti requirement anti-leakage brief tidak penting -- justru ini
menunjukkan leakage bisa MEMBERI HASIL YANG BAGUS SECARA KEBETULAN (atau tidak berpengaruh) pada
dataset besar, TAPI risikonya jauh lebih serius pada dataset kecil, atau ketika test set punya
distribusi yang berbeda signifikan dari train (bukan kasus di sini karena stratified split).
Metodologi yang benar (fit hanya di train) tetap wajib diikuti secara prinsip, terlepas dari apakah
dampak praktisnya besar atau kecil untuk kasus spesifik ini -- karena kita tidak bisa tahu
di muka seberapa besar dampaknya tanpa eksperimen kontrol seperti ini.

**Konteks perbandingan dengan paper:** baik versi replikasi murni maupun terkoreksi SAMA-SAMA masih meleset
jauh dari paper untuk Precision/F1 kelas-positif murni (pola yang sama seperti di notebook 05) --
mengonfirmasi ulang bahwa gap besar itu soal definisi metrik (kelas-positif vs weighted average),
BUKAN soal leakage. Baik versi replikasi murni maupun terkoreksi mendekati paper kalau dilihat dari
Precision(weighted)/F1(weighted).
